# 🏥 SVM Applied — Breast Cancer Dataset
### Atlas Skilltech University — Machine Learning Course
---
**Is notebook mein hum sikhenge:**
- Real medical dataset par SVM kaise apply karte hain
- Complete pipeline: Data Loading → EDA → Preprocessing → Model → Evaluation
- Breast Cancer dataset — ek life-saving classification problem
- SVM Capstone Project

**Breast Cancer Dataset kya hai?**
> Is dataset mein breast tumor ki measurements hain.
> **Goal:** Predict karo tumor **Malignant** (Cancer) hai ya **Benign** (Safe)
>
> - **Malignant (0):** Cancerous tumor — dangerous!
> - **Benign (1):** Non-cancerous tumor — safe
>
> Yeh ek real life-saving application hai! Galat prediction = patient ki jaan khatre mein.
> Isliye Recall zyada important hai yahan (koi cancer miss nahi hona chahiye).

## Step 1: Libraries Import Karo
---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import SVC
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, accuracy_score,
    roc_curve, roc_auc_score, RocCurveDisplay
)

import warnings
warnings.filterwarnings('ignore')

print("✅ Saari libraries import ho gayi!")

## Step 2: Data Loading — Dataset Load Karo
---
**Keyword Explanation:**
- **load_breast_cancer()** — sklearn ka built-in medical dataset
- **569 samples** — 569 patients ka data
- **30 features** — tumor measurements (radius, texture, perimeter, area, etc.)
- **Binary Classification** — Malignant (0) ya Benign (1)

In [ ]:
# Breast Cancer dataset load karo
cancer = load_breast_cancer()

df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df['target'] = cancer.target
df['diagnosis'] = df['target'].map({0: 'Malignant', 1: 'Benign'})

print("Dataset ka shape:", df.shape)
print("\nFeatures (pehli 5):")
print(list(cancer.feature_names[:5]))
print(f"\nTotal Features: {len(cancer.feature_names)}")
print("\nTarget Distribution:")
print(df['diagnosis'].value_counts())
print("\nPehli 3 rows:")
df.head(3)

## Step 3: EDA — Exploratory Data Analysis
---
**EDA mein kya dekhenge:**
1. Class distribution (Malignant vs Benign kitne hain?)
2. Missing values check
3. Important features ka distribution
4. Correlation heatmap
5. Boxplots — class ke hisaab se features

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['diagnosis'].value_counts()
colors = ['#E74C3C', '#2ECC71']

axes[0].bar(counts.index, counts.values, color=colors, edgecolor='black', width=0.5)
axes[0].set_title('Tumor Distribution', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count')
for i, (idx, val) in enumerate(counts.items()):
    axes[0].text(i, val + 3, str(val), ha='center', fontsize=12, fontweight='bold')

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, explode=[0.05, 0])
axes[1].set_title('Percentage Distribution', fontsize=12, fontweight='bold')

plt.suptitle('Breast Cancer Dataset — Class Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Missing Values:", df.isnull().sum().sum())
print("Observation: Benign (safe) zyada hain — dataset thoda imbalanced hai.")

In [ ]:
# Important features ka distribution by diagnosis
important_features = ['mean radius', 'mean texture', 'mean perimeter', 
                       'mean area', 'mean smoothness', 'mean concavity']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(important_features):
    benign = df[df['target']==1][feat]
    malignant = df[df['target']==0][feat]
    
    axes[i].hist(malignant, bins=25, alpha=0.7, color='#E74C3C', label='Malignant (Cancer)', edgecolor='black')
    axes[i].hist(benign, bins=25, alpha=0.7, color='#2ECC71', label='Benign (Safe)', edgecolor='black')
    axes[i].set_title(feat, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')
    axes[i].legend(fontsize=8)

plt.suptitle('Feature Distribution — Malignant vs Benign', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("Dekho: Mean Radius, Perimeter, Area mein Malignant clearly zyada large hai!")

In [ ]:
# Correlation Heatmap — top 10 features
top_features = ['mean radius', 'mean texture', 'mean perimeter', 'mean area',
                 'mean smoothness', 'mean compactness', 'mean concavity',
                 'mean concave points', 'mean symmetry', 'mean fractal dimension']

plt.figure(figsize=(12, 9))
corr = df[top_features + ['target']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            linewidths=0.5, annot_kws={'size': 8}, mask=mask)
plt.title('Correlation Heatmap — Top 10 Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("Observation: Mean radius, perimeter, area mein bahut high correlation (0.98+)!")
print("Yeh multicollinearity hai — SVM isko handle kar sakta hai.")

In [ ]:
# Boxplot — class ke hisaab se
box_features = ['mean radius', 'mean area', 'mean concavity', 'mean concave points']

fig, axes = plt.subplots(1, 4, figsize=(16, 5))

for ax, feat in zip(axes, box_features):
    data_plot = [df[df['target']==0][feat], df[df['target']==1][feat]]
    bp = ax.boxplot(data_plot, patch_artist=True, labels=['Malignant', 'Benign'])
    bp['boxes'][0].set_facecolor('#E74C3C')
    bp['boxes'][1].set_facecolor('#2ECC71')
    ax.set_title(feat, fontsize=10, fontweight='bold')
    ax.set_ylabel('Value')

plt.suptitle('Boxplot — Features by Diagnosis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 4: Preprocessing
---
**Keyword Explanation:**
- **30 features** hai is dataset mein — SVM ke liye scaling ZARURI hai
- **stratify** — ensure karo ki Malignant/Benign dono train aur test mein equally divided ho
- **StandardScaler** — har feature ka mean=0, std=1 kar do

In [ ]:
# Features aur Target
X = cancer.data   # 30 features
y = cancer.target  # 0=Malignant, 1=Benign

print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape:   {y.shape}")

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set: {X_train.shape[0]} samples")
print(f"Testing set:  {X_test.shape[0]} samples")

# Feature Scaling — 30 features hain, scaling BAHUT important hai
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n✅ Scaling complete! 30 features ab ek hi scale par hain.")

## Step 5: Model Training — Kernels Compare Karo
---
**Keyword Explanation:**
- **kernel** — kaunsi transformation use karni hai
- **C** — regularization strength (tradeoff: margin size vs misclassification)
- **gamma** — RBF kernel ki bandwidth (auto ya manual set)
- **probability=True** — ROC curve ke liye probability scores chahiye

In [ ]:
# Alag Kernels compare karo
print("Kernel Comparison — Breast Cancer Dataset")
print("-" * 65)
print(f"{'Kernel':<30} {'Train Acc':<12} {'Test Acc':<12} {'Support Vectors'}")
print("-" * 65)

kernel_configs = [
    ('Linear', SVC(kernel='linear', C=1.0)),
    ('RBF (C=1, gamma=scale)', SVC(kernel='rbf', C=1.0, gamma='scale')),
    ('RBF (C=10, gamma=scale)', SVC(kernel='rbf', C=10.0, gamma='scale')),
    ('Polynomial (d=2)', SVC(kernel='poly', C=1.0, degree=2)),
    ('Polynomial (d=3)', SVC(kernel='poly', C=1.0, degree=3)),
]

best_model = None
best_acc = 0

for name, model in kernel_configs:
    model.fit(X_train_scaled, y_train)
    train_acc = model.score(X_train_scaled, y_train)
    test_acc = model.score(X_test_scaled, y_test)
    print(f"{name:<30} {train_acc:.4f}      {test_acc:.4f}      {len(model.support_vectors_)}")
    if test_acc > best_acc:
        best_acc = test_acc
        best_model = model

In [ ]:
# Best Model — RBF SVM train karo
svm_best = SVC(kernel='rbf', C=10.0, gamma='scale', probability=True, random_state=42)
svm_best.fit(X_train_scaled, y_train)

y_pred = svm_best.predict(X_test_scaled)

print("Best SVM Model (RBF, C=10):")
print(f"  Training Accuracy: {svm_best.score(X_train_scaled, y_train)*100:.2f}%")
print(f"  Testing Accuracy:  {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"  Support Vectors:   {len(svm_best.support_vectors_)}")

## Step 6: Confusion Matrix — Purana aur Naya Tarika
---

In [ ]:
cm = confusion_matrix(y_test, y_pred)
labels = ['Malignant\n(Cancer)', 'Benign\n(Safe)']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Purana tarika — Seaborn Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=['Malignant', 'Benign'],
            yticklabels=['Malignant', 'Benign'],
            annot_kws={'size': 16, 'weight': 'bold'},
            linewidths=0.5, ax=axes[0])
axes[0].set_title('Purana Tarika: Seaborn Heatmap', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted (Model ne kya bola)', fontsize=10)
axes[0].set_ylabel('Actual (Sach mein kya tha)', fontsize=10)

# Naya tarika — ConfusionMatrixDisplay
cmd = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Malignant', 'Benign'])
cmd.plot(cmap='Greens', colorbar=False, ax=axes[1])
axes[1].set_title('Naya Tarika: ConfusionMatrixDisplay', fontsize=12, fontweight='bold')

plt.suptitle('SVM Confusion Matrix — Breast Cancer', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"\nConfusion Matrix Analysis:")
print(f"  True Positive (Cancer sahi pakda):    {tp}")
print(f"  True Negative (Safe sahi bola):       {tn}")
print(f"  False Positive (Safe ko Cancer bola): {fp} ← Type I Error")
print(f"  False Negative (Cancer ko Safe bola): {fn} ← Type II Error (DANGEROUS!)")

## Step 7: Precision, Recall, F1 Score
---
**Medical Context mein Recall zyada important kyun hai?**
> - **False Negative (Cancer miss karna)** = Patient ko cancer hai lekin model ne bola nahi = **LIFE THREATENING!**
> - **False Positive (Galat Cancer detect karna)** = Patient ko cancer nahi, fir bhi alert = More tests karne padte hain, stressful lekin treatable
>
> Isliye Recall (Sensitivity) yahan priority hai!

In [ ]:
print("=" * 65)
print("CLASSIFICATION REPORT — BREAST CANCER SVM")
print("=" * 65)
print(classification_report(y_test, y_pred, target_names=['Malignant', 'Benign']))

print("\n⚠️  Medical Context mein:")
print("   Recall for Malignant = kitne actual cancer cases pakde?")
print("   Yeh HIGHEST possible hona chahiye!")

In [ ]:
# C parameter ka Precision/Recall par asar
from sklearn.metrics import precision_score, recall_score, f1_score

c_values = [0.01, 0.1, 1, 10, 100, 1000]
precisions, recalls, f1s, test_accs = [], [], [], []

for C in c_values:
    svm_c = SVC(kernel='rbf', C=C, gamma='scale')
    svm_c.fit(X_train_scaled, y_train)
    y_p = svm_c.predict(X_test_scaled)
    precisions.append(precision_score(y_test, y_p, pos_label=0))
    recalls.append(recall_score(y_test, y_p, pos_label=0))
    f1s.append(f1_score(y_test, y_p, pos_label=0))
    test_accs.append(accuracy_score(y_test, y_p))

plt.figure(figsize=(12, 5))
plt.plot(c_values, precisions, 'b-o', label='Precision (Malignant)', linewidth=2)
plt.plot(c_values, recalls, 'r-s', label='Recall (Malignant)', linewidth=2)
plt.plot(c_values, f1s, 'g-^', label='F1 Score (Malignant)', linewidth=2)
plt.plot(c_values, test_accs, 'k--d', label='Overall Accuracy', linewidth=2)
plt.xscale('log')
plt.xlabel('C (Regularization Parameter) — Log Scale', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.title('C Parameter ka Precision/Recall/F1 par Asar', fontsize=13, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

## Step 8: ROC Curve — Purana aur Naya Tarika
---
**Keyword Explanation:**
- **ROC Curve** — model ki performance at different thresholds
- **AUC** — Area Under Curve (1.0 = perfect, 0.5 = random)
- **TPR** = Recall = True Positive Rate
- **FPR** = False Positive Rate

In [ ]:
y_prob = svm_best.predict_proba(X_test_scaled)[:, 0]  # Malignant probability

fpr, tpr, thresholds = roc_curve(y_test, y_prob, pos_label=0)
auc_score = roc_auc_score(y_test, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Purana tarika — manual roc_curve
axes[0].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {auc_score:.4f})')
axes[0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
axes[0].fill_between(fpr, tpr, alpha=0.2, color='darkorange')
axes[0].set_xlabel('False Positive Rate (FPR)', fontsize=11)
axes[0].set_ylabel('True Positive Rate (TPR / Recall)', fontsize=11)
axes[0].set_title('Purana Tarika: Manual ROC Curve', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Naya tarika — RocCurveDisplay
rcd = RocCurveDisplay.from_estimator(
    svm_best, X_test_scaled, y_test, pos_label=0, ax=axes[1],
    name='SVM RBF (C=10)'
)
axes[1].plot([0, 1], [0, 1], 'k--', label='Random')
axes[1].set_title('Naya Tarika: RocCurveDisplay', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('ROC Curve — Breast Cancer SVM', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"AUC Score: {auc_score:.4f}")
print("AUC 0.9+ = Excellent model! Yeh medical diagnosis ke liye bahut achha hai.")

## Step 9: GridSearchCV — Best Hyperparameters Dhundo
---
**Keyword Explanation:**
- **GridSearchCV** — systematically har combination try karo aur best dhundo
- **param_grid** — dictionary of parameters to try
- **cv=5** — 5-fold cross validation use karo har combination ke liye
- **scoring** — kaunsa metric optimize karna hai

In [ ]:
# GridSearchCV — Best C aur Gamma dhundo
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01],
    'kernel': ['rbf']
}

print("GridSearchCV chal raha hai — thoda waqt lagega...")
grid_search = GridSearchCV(
    SVC(probability=True),
    param_grid,
    cv=5,
    scoring='recall',  # Recall maximize karo (medical ke liye)
    n_jobs=-1,
    verbose=0
)
grid_search.fit(X_train_scaled, y_train)

print(f"\nBest Parameters: {grid_search.best_params_}")
print(f"Best CV Recall: {grid_search.best_score_:.4f}")

best_svm = grid_search.best_estimator_
y_pred_best = best_svm.predict(X_test_scaled)
print(f"Test Accuracy: {accuracy_score(y_test, y_pred_best)*100:.2f}%")

## Step 10: Final Performance Summary
---

In [ ]:
# Final Summary
print("=" * 65)
print("FINAL PERFORMANCE SUMMARY — BREAST CANCER SVM")
print("=" * 65)
print(f"Dataset: Breast Cancer Wisconsin")
print(f"Samples: {len(y)} | Features: {X.shape[1]}")
print(f"Training: {len(y_train)} | Testing: {len(y_test)}")
print(f"Best Kernel: RBF | Best C: 10 | Gamma: scale")
print()
print(f"Training Accuracy: {svm_best.score(X_train_scaled, y_train)*100:.2f}%")
print(f"Testing Accuracy:  {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"AUC Score:         {roc_auc_score(y_test, svm_best.predict_proba(X_test_scaled)[:, 0]):.4f}")
print(f"Support Vectors:   {len(svm_best.support_vectors_)}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Malignant', 'Benign']))
print("=" * 65)

In [ ]:
# Naya patient predict karo!
print("Naya Patient Diagnosis Karo!")
print("=" * 40)

# Random ek test sample lo
sample_idx = 5
sample = X_test[sample_idx:sample_idx+1]
sample_scaled = scaler.transform(sample)

prediction = svm_best.predict(sample_scaled)
probability = svm_best.predict_proba(sample_scaled)

diagnosis = 'MALIGNANT (Cancer!)' if prediction[0] == 0 else 'BENIGN (Safe)'
print(f"\nPrediction: {diagnosis}")
print(f"Confidence: Malignant={probability[0][0]*100:.1f}% | Benign={probability[0][1]*100:.1f}%")
print(f"Actual:     {'MALIGNANT' if y_test[sample_idx]==0 else 'BENIGN'}")
print(f"Correct:    {'✅ YES' if prediction[0]==y_test[sample_idx] else '❌ NO'}")

## SVM Capstone Project 🎯
---
**Tumhara Assignment — 3 Parts:**

**Part A — Kernel War:**
1. Linear, RBF, Polynomial (d=2,3,4), Sigmoid kernels compare karo
2. Accuracy, Precision, Recall, F1 ek table mein dikhao
3. Medical context mein best kernel kaunsa hai aur kyun? Explain karo.

**Part B — C aur Gamma Tuning:**
1. GridSearchCV use karo C=[0.001, 0.01, 0.1, 1, 10, 100, 1000] aur gamma=['scale', 0.001, 0.01, 0.1, 1]
2. Best combination dhundo
3. Heatmap plot karo (C vs Gamma vs Score)

**Part C — Feature Selection:**
1. Sirf top 10 features use karke model banao (by correlation with target)
2. Full 30 features vs Top 10 features compare karo
3. Kya performance badhi ya ghati?

**Bonus:**
- Precision-Recall curve plot karo (Logistic Regression se compare karo)
- Class weights adjust karo (`class_weight='balanced'`) — kya recall improve hota hai?

In [ ]:
# Part A — Yahan apna code likho
print("Part A: Kernel War — Comparison Table")
print("Yahan Apna Code Likho! ⚔️")

# Hint:
# kernels_to_test = ['linear', 'rbf', ('poly', 2), ('poly', 3), ('poly', 4), 'sigmoid']
# results = []
# for kernel in kernels_to_test:
#     ...

In [ ]:
# Part B — GridSearchCV Heatmap
print("Part B: C aur Gamma Tuning")
print("Yahan Apna Code Likho! 🔍")

# Hint:
# results_df = pd.DataFrame(grid_search.cv_results_)
# pivot = results_df.pivot_table(values='mean_test_score', index='param_C', columns='param_gamma')
# sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd')

In [ ]:
# Part C — Feature Selection
print("Part C: Top 10 Features vs All 30 Features")
print("Yahan Apna Code Likho! 📊")

# Hint:
# correlation = pd.Series(np.corrcoef(X.T, y)[:-1, -1], index=cancer.feature_names)
# top10 = abs(correlation).nlargest(10).index.tolist()
# X_top10 = df[top10].values
# ...

---
## Notebook Complete! 🎉

**Aaj humne sikha:**

| Topic | Kya Sikha |
|-------|----------|
| Data Loading | sklearn breast cancer dataset, 30 features |
| EDA | Distribution, Heatmap, Boxplot — Malignant vs Benign |
| Preprocessing | StandardScaler, Stratified Train-Test Split |
| Kernel Comparison | Linear, RBF, Poly compare kiya |
| C Parameter | Regularization ka effect on precision/recall |
| Confusion Matrix | Dono methods — seaborn aur sklearn |
| ROC Curve | Manual aur RocCurveDisplay dono |
| GridSearchCV | Systematic hyperparameter tuning |
| Medical Context | Recall > Precision kyun? Life-or-death decisions |

**Next Step:** SVM Capstone Project complete karo! 💪

---
*Atlas Skilltech University — Machine Learning Course*